In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('ggplot')
params = {'legend.fontsize': 'medium',
        'figure.figsize': (20, 12),
        'axes.labelsize': 'medium',
        'axes.titlesize':'medium',
        'xtick.labelsize':'medium',
        'ytick.labelsize':'medium'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import rateslib as rl
import QuantLib as ql

import time
import datetime
import pytz

NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
LDN_tz = pytz.timezone("Europe/London") 
UTC_tz = pytz.timezone("UTC") 

import sys
sys.path.append("../../")

from RVUtils.plt_timeseries import make_secondary_axis_plot

In [12]:
from MDP.USTFutures.USTFuturesMDP import USTFuturesMDP
from MDP.FixedRateBonds.FixedRateBondsMDP import FixedRateBondsMDP

In [8]:
ts =  CHI_tz.localize(datetime.datetime(2026, 3, 24, 14, 0))
symbol = "USM26"  

mdp = USTFuturesMDP(source="BARCHART_USTF-RL")

pricer = mdp.get_pricer(
    {
        "symbols": [symbol],
        "timestamp": ts,
        "include_basket": True,
    }
)[symbol]

fut = pricer.build_pricable()
fut_price = pricer.price(fut)
fut_ytm = pricer.yield_to_maturity(fut)

ctd = pricer.ctd()
ctd_meta = (ctd.meta() or {}) if ctd else {}
ctd_cusip = ctd_meta.get("cusip")

In [9]:
basket = mdp.get_delivery_basket(
    as_of=ts.date(),
    symbol=symbol,
    source="RL_CME_TCF",
)

rows = []
for bond_pricer, cf in zip(basket["basket_pricers"], basket["conversion_factors"]):
    meta = bond_pricer.meta() or {}
    clean_price = float(bond_pricer.clean_price())
    gross_basis = clean_price - fut_price * float(cf)

    rows.append(
        {
            "cusip": meta.get("cusip"),
            "label": meta.get("label"),
            "clean_price": clean_price,
            "invoice_cf": float(cf),
            "gross_basis": gross_basis,
            "is_ctd": meta.get("cusip") == ctd_cusip,
        }
    )

basis_df = pd.DataFrame(rows).sort_values(["is_ctd", "gross_basis"], ascending=[False, True]).reset_index(drop=True)

print(f"{symbol} futures price: {fut_price:.4f}")
print(f"{symbol} futures YTM:   {fut_ytm:.4f}")
print(f"CTD: {ctd_meta.get('label')} ({ctd_meta.get('cusip')})")

USM26 futures price: 112.5938
USM26 futures YTM:   4.9762
CTD: T 4 1/2 Feb 44 (912810TZ1)


In [10]:
basis_df

,cusip,label,clean_price,invoice_cf,gross_basis,is_ctd
0,912810TZ1,T 4 1/2 Feb 44,94.734934,0.8388,0.291296,True
1,912810RD2,T 3 3/4 Nov 43,85.841766,0.7602,0.247997,False
2,912810RC4,T 3 5/8 Aug 43,84.617178,0.7491,0.273200,False
3,912810RE0,T 3 5/8 Feb 44,84.139840,0.7448,0.280015,False
4,912810TM0,T 4 Nov 42,89.692382,0.7941,0.281685,False
5,912810TU2,T 4 3/8 Aug 43,93.547556,0.8283,0.286153,False
6,912810RB6,T 2 7/8 May 43,76.028899,0.6726,0.298343,False
7,912810TQ1,T 3 7/8 Feb 43,88.064137,0.7794,0.308568,False
8,912810UB2,T 4 5/8 May 44,96.126890,0.8510,0.309609,False
9,912810TS7,T 3 7/8 May 43,87.834548,0.7773,0.315426,False


In [15]:
cash_mdp = FixedRateBondsMDP(source="USTS_FEDINVEST_WSJ_LIVE-RL")
pricers = cash_mdp.get_pricer({
    "cusips": ["912810TZ1"],
    "timestamp": "live",
})
pr = next(iter(pricers.values())) 

print("YTM:", pr.ytm())
print("Clean price:", pr.clean_price())
print("Dirty price:", pr.dirty_price())
print("Maturity:", pr.maturity_date())
print("Meta:", pr.meta())

YTM: 4.933
Clean price: 94.88303454492454
Dirty price: 95.35541023553228
Maturity: 2044-02-15
Meta: {'record_date': datetime.date(2024, 2, 29), 'label': 'T 4 1/2 Feb 44', 'cusip': '912810TZ1', 'oi': '20-Year', 'auction_date': datetime.date(2024, 2, 21), 'issue_date': datetime.date(2024, 2, 29), 'maturity_date': datetime.date(2044, 2, 15), 'cpn': 4.5, 'rank': 8, 'timestamp': datetime.datetime(2026, 3, 24, 10, 55, 43, tzinfo=<DstTzInfo 'America/New_York' EDT-1 day, 20:00:00 DST>)}
